In [2]:
from pyscf import gto, scf, cc
import numpy as np

a = 2.5 # 2aB
nH = 2
atoms = ""
for i in range(nH):
    atoms += f"H {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="6-31g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.UHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]
# et = mycc.ccsd_t()
# print(mycc.e_tot + et)

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-19-generic', version='#19~24.04.2-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 23:08:46 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Tue Mar 17 17:39:40 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

np.float64(-0.04120276824601407)

In [5]:
# example for PT2
options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 1,
            'seed': 17,
            'walker_type': 'uhf',
            'trial': 'uccsd_pt2',
            'dt':0.005,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (1, 1)
# Number of basis functions: 4
# Number of Cholesky vectors: 10
#


In [6]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
# ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
# ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = e_init
# eci = trial.calc_energy(
#     prop_data['walkers'], ham_data, wave_data)
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# norb: 4
# nelec: (1, 1)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 1
# seed: 17
# walker_type: uhf
# trial: uccsd_pt2
# dt: 0.005
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# symmetry: False
# save_walkers: False
# n_batch: 1
# max_error: 0.001
-1.0361167077357236
0.0


In [7]:
from jax import jit, vmap, lax
import opt_einsum as oe

@partial(jit, static_argnums=0)
def _calc_energy_pt(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
) -> complex:
    nocc_a, t2_aa = self.nelec[0], wave_data["t2aa"]
    nocc_b, t2_bb = self.nelec[1], wave_data["t2bb"]
    t2_ab = wave_data["t2ab"]
    mo_a, mo_b = wave_data['mo_ta'], wave_data['mo_tb']
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)
    h1_a = ham_data["h1"][0]
    h1_b = ham_data["h1"][1]

    # full green's function G_pq
    green_a = (walker_up @ (jnp.linalg.inv(mo_a.T @ walker_up)) @ mo_a.T).T
    green_b = (walker_dn @ (jnp.linalg.inv(mo_b.T @ walker_dn)) @ mo_b.T).T
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    hg = hg_a + hg_b # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>
    # one body energy
    e1_0 = hg

    # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>
    # double excitations
    t2g_a = oe.contract("iajb,ia->jb", t2_aa, green_a[:nocc_a,nocc_a:],
                        backend="jax") / 4
    t2g_b = oe.contract("iajb,ia->jb", t2_bb, green_b[:nocc_b,nocc_b:], 
                        backend="jax") / 4
    t2g_ab_a = oe.contract("iajb,jb->ia", t2_ab, green_b[:nocc_b,nocc_b:],
                            backend="jax")
    t2g_ab_b = oe.contract("iajb,ia->jb", t2_ab, green_a[:nocc_a,nocc_a:],
                            backend="jax")
    # t_iajb (G_ia G_jb - G_ib G_ja)
    gt2g_a = oe.contract("jb,jb->", t2g_a, green_a[:nocc_a,nocc_a:], 
                        backend="jax")
    gt2g_b = oe.contract("jb,jb->", t2g_b, green_b[:nocc_b,nocc_b:], 
                        backend="jax")
    gt2g_ab = oe.contract("ia,ia->", t2g_ab_a, green_a[:nocc_a,nocc_a:], 
                            backend="jax")
    gt2g = 2 * (gt2g_a + gt2g_b) + gt2g_ab # <exp(T1)HF|T2|walker>/<exp(T1)HF|walker>

    e1_2_1 = hg * gt2g
    
    t2_green_a = (greenp_a @ t2g_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    t2_green_ab_a = (greenp_a @ t2g_ab_a.T) @ green_a[:nocc_a,:]
    t2_green_b = (greenp_b @ t2g_b.T) @ green_b[:nocc_b,:]
    t2_green_ab_b = (greenp_b @ t2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract(
        "pq,pq->", h1_a, 4 * t2_green_a + t2_green_ab_a, backend="jax")
    e1_2_2_b = -oe.contract(
        "pq,pq->", h1_b, 4 * t2_green_b + t2_green_ab_b, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>
    # double excitations
    # e2_2_1 = e2_0 * gt2g
    lg_a = oe.contract("gpq,pq->g", chol_a, green_a, backend="jax")
    lg_b = oe.contract("gpq,pq->g", chol_b, green_b, backend="jax")
    lt2g_a = oe.contract("gpq,pq->g",
                        chol_a, 8 * t2_green_a + 2 * t2_green_ab_a,
                        backend="jax")
    lt2g_b = oe.contract("gpq,pq->g",
        chol_b, 8 * t2_green_b + 2 * t2_green_ab_b,
        backend="jax")
    e2_2_2_1 = -((lt2g_a + lt2g_b) @ (lg_a + lg_b)) / 2.0

    def scanned_fun(carry, x):
        chol_a_i, chol_b_i = x
        # e2_0
        lg_a_i = oe.contract("pr,qr->pq", chol_a_i, green_a, backend="jax")
        lg_b_i = oe.contract("pr,qr->pq", chol_b_i, green_b, backend="jax")
        e2_0_1_i = (jnp.trace(lg_a_i) + jnp.trace(lg_b_i))**2 / 2.0
        e2_0_2_i = -(oe.contract('pq,qp->',lg_a_i,lg_a_i, backend="jax") 
                    + oe.contract('pq,qp->',lg_b_i,lg_b_i, backend="jax")
                    ) / 2.0
        carry[0] += e2_0_1_i + e2_0_2_i
        # e2_2
        gl_a_i = oe.contract("pr,rq->pq", green_a, chol_a_i,
                            backend="jax")
        gl_b_i = oe.contract("pr,rq->pq", green_b, chol_b_i,
                            backend="jax")
        lt2_green_a_i = oe.contract(
            "pr,qr->pq", chol_a_i, 8 * t2_green_a + 2 * t2_green_ab_a,
            backend="jax")
        lt2_green_b_i = oe.contract(
            "pr,qr->pq", chol_b_i, 8 * t2_green_b + 2 * t2_green_ab_b,
            backend="jax")
        carry[1] += 0.5 * (
            oe.contract("pq,pq->", gl_a_i, lt2_green_a_i, backend="jax")
            + oe.contract("pq,pq->", gl_b_i, lt2_green_b_i, backend="jax")
        )
        glgp_a_i = oe.contract(
            "iq,qa->ia", gl_a_i[:nocc_a,:], greenp_a, backend="jax"
        )
        glgp_b_i = oe.contract(
            "iq,qa->ia", gl_b_i[:nocc_b,:], greenp_b, backend="jax"
        )
        l2t2_a = 0.5 * oe.contract(
            "ia,jb,iajb->",glgp_a_i,glgp_a_i,t2_aa,
            backend="jax")
        l2t2_b = 0.5 * oe.contract(
            "ia,jb,iajb->",glgp_b_i,glgp_b_i,t2_bb,
            backend="jax")
        l2t2_ab = oe.contract(
            "ia,jb,iajb->",glgp_a_i,glgp_b_i,t2_ab,
            backend="jax")
        carry[2] += l2t2_a + l2t2_b + l2t2_ab
        return carry, 0.0

    [e2_0, e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0, 0.0, 0.0], (chol_a, chol_b))
    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>

    o0 = jnp.linalg.det(walker_up[:nocc_a,:nocc_a]
        ) * jnp.linalg.det(walker_dn[:nocc_b,:nocc_b])
    # <exp(T1)HF|walker>/<HF|walker>
    t1 = jnp.linalg.det(wave_data["mo_ta"].T.conj() @ walker_up
        ) * jnp.linalg.det(wave_data["mo_tb"].T.conj() @ walker_dn) / o0
    t2 = gt2g * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>

    return t1, t2, e0, e1

In [ ]:
@partial(jit, static_argnums=0)
def _calc_energy_pt_faster(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
) -> complex:
    # CISD trial with half Green
    nocc_a, nocc_b = self.nelec
    norb = self.norb
    nvir_a = norb - nocc_a
    nvir_b = norb - nocc_b
    t2_aa = wave_data["rot_t2aa"]
    t2_bb = wave_data["rot_t2bb"]
    t2_ab = wave_data["rot_t2ab"]
    mo_a, mo_b = wave_data['mo_ta'], wave_data['mo_tb']
    h1_a, h1_b = ham_data["rot_h1"]
    dh1_a, dh1_b = ham_data["d_h1"] # delta_pb h_pq
    chol_a = ham_data["rot_chol"][0].reshape(-1, nocc_a, norb)
    chol_b = ham_data["rot_chol"][1].reshape(-1, nocc_b, norb)
    dchol_a = ham_data["d_chol"][0].reshape(-1, nvir_a, norb) # delta_pb L_gpq
    dchol_b = ham_data["d_chol"][1].reshape(-1, nvir_b, norb)

    # half green's function G_pq
    green_a = (walker_up @ (jnp.linalg.inv(mo_a.T @ walker_up))).T # G_lq
    green_b = (walker_dn @ (jnp.linalg.inv(mo_b.T @ walker_dn))).T
    # greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    # greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    # ref one-body energy
    hg_a = oe.contract("pq,rq->pr", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,rq->pr", h1_b, green_b, backend="jax")
    trhg_a = oe.contract("pp->", hg_a, backend="jax")
    trhg_b = oe.contract("pp->", hg_b, backend="jax")
    e1_0 = trhg_a + trhg_b # <psi|h1|walker>/<psi|walker>

    # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>
    # double excitations
    t2g_aa_a = oe.contract("iajb,ia->jb", t2_aa, green_a[:nocc_a,nocc_a:], backend="jax") / 4
    t2g_bb_b = oe.contract("iajb,ia->jb", t2_bb, green_b[:nocc_b,nocc_b:], backend="jax") / 4
    t2g_ab_a = oe.contract("iajb,jb->ia", t2_ab, green_b[:nocc_b,nocc_b:], backend="jax")
    t2g_ab_b = oe.contract("iajb,ia->jb", t2_ab, green_a[:nocc_a,nocc_a:], backend="jax")
    # t_iajb (G_ia G_jb - G_ib G_ja)
    gt2g_a = oe.contract("jb,jb->", t2g_aa_a, green_a[:nocc_a,nocc_a:], backend="jax")
    gt2g_b = oe.contract("jb,jb->", t2g_bb_b, green_b[:nocc_b,nocc_b:], backend="jax")
    gt2g_ab = oe.contract("ia,ia->", t2g_ab_a, green_a[:nocc_a,nocc_a:], backend="jax")
    gt2g = 2 * (gt2g_a + gt2g_b) + gt2g_ab # <exp(T1)HF|T2|walker>/<exp(T1)HF|walker>

    e1_2_1 = e1_0 * gt2g

    # Gp_pb h_pq G_jq
    ghg_a = oe.contract('pb,pj->jb', green_a[:nocc_a, nocc_a:], hg_a[:,:nocc_a], backend="jax")
    dhg_a = oe.contract('jq,bq->jb', dh1_a, green_a[:nocc_a,:], backend="jax")
    gphg_a = ghg_a - dhg_a
    
    # t2_green_a = (greenp_a @ t2g_a.T) @ green_a[:nocc_a,:] # t_iajb G_ia G_jq Gp_pb
    t2_green_a = oe.contract('pb,jb,jq->pq', green_a[:nocc_a,nocc_a:], t2g_aa_a, green_a[:nocc_a,:], backend="jax") # t_iajb G_ia G_jq G_pb
    t2_green_ab_a = oe.contract('pb,jb,jq->pq', green_a[:nocc_a,nocc_a:], t2g_ab_a, green_a[:nocc_a,:], backend="jax") # (greenp_a @ t2g_ab_a.T) @ green_a[:nocc_a,:]
    t2_green_b = oe.contract('pb,jb,jq->pq', green_b[:nocc_b,nocc_b:], t2g_bb_b, green_b[:nocc_b,:], backend="jax")
    t2_green_ab_b = oe.contract('pb,jb,jq->pq', green_b[:nocc_b,nocc_b:], t2g_ab_b, green_b[:nocc_b,:], backend="jax") # (greenp_b @ t2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract("pq,pq->", h1_a, 4 * t2_green_a + t2_green_ab_a, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, 4 * t2_green_b + t2_green_ab_b, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>
    # double excitations
    # e2_2_1 = e2_0 * gt2g
    lg_a = oe.contract("gpq,pq->g", chol_a, green_a, backend="jax")
    lg_b = oe.contract("gpq,pq->g", chol_b, green_b, backend="jax")
    lt2g_a = oe.contract("gpq,pq->g", chol_a, 8 * t2_green_a + 2 * t2_green_ab_a, backend="jax")
    lt2g_b = oe.contract("gpq,pq->g", chol_b, 8 * t2_green_b + 2 * t2_green_ab_b, backend="jax")
    e2_2_2_1 = -((lt2g_a + lt2g_b) @ (lg_a + lg_b)) / 2.0

    def scanned_fun(carry, x):
        chol_a_i, chol_b_i = x
        # e2_0
        lg_a_i = oe.contract("pr,qr->pq", chol_a_i, green_a, backend="jax")
        lg_b_i = oe.contract("pr,qr->pq", chol_b_i, green_b, backend="jax")
        e2_0_1_i = (jnp.trace(lg_a_i) + jnp.trace(lg_b_i))**2 / 2.0
        e2_0_2_i = -(oe.contract('pq,qp->',lg_a_i,lg_a_i, backend="jax") 
                    + oe.contract('pq,qp->',lg_b_i,lg_b_i, backend="jax")
                    ) / 2.0
        carry[0] += e2_0_1_i + e2_0_2_i
        # e2_2
        gl_a_i = oe.contract("pr,rq->pq", green_a, chol_a_i,
                            backend="jax")
        gl_b_i = oe.contract("pr,rq->pq", green_b, chol_b_i,
                            backend="jax")
        lt2_green_a_i = oe.contract(
            "pr,qr->pq", chol_a_i, 8 * t2_green_a + 2 * t2_green_ab_a,
            backend="jax")
        lt2_green_b_i = oe.contract(
            "pr,qr->pq", chol_b_i, 8 * t2_green_b + 2 * t2_green_ab_b,
            backend="jax")
        carry[1] += 0.5 * (
            oe.contract("pq,pq->", gl_a_i, lt2_green_a_i, backend="jax")
            + oe.contract("pq,pq->", gl_b_i, lt2_green_b_i, backend="jax")
        )
        glgp_a_i = oe.contract(
            "iq,qa->ia", gl_a_i[:nocc_a,:], greenp_a, backend="jax"
        )
        glgp_b_i = oe.contract(
            "iq,qa->ia", gl_b_i[:nocc_b,:], greenp_b, backend="jax"
        )
        l2t2_a = 0.5 * oe.contract(
            "ia,jb,iajb->",glgp_a_i,glgp_a_i,t2_aa,
            backend="jax")
        l2t2_b = 0.5 * oe.contract(
            "ia,jb,iajb->",glgp_b_i,glgp_b_i,t2_bb,
            backend="jax")
        l2t2_ab = oe.contract(
            "ia,jb,iajb->",glgp_a_i,glgp_b_i,t2_ab,
            backend="jax")
        carry[2] += l2t2_a + l2t2_b + l2t2_ab
        return carry, 0.0

    [e2_0, e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0, 0.0, 0.0], (chol_a, chol_b))
    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>

    o0 = jnp.linalg.det(walker_up[:nocc_a,:nocc_a]
        ) * jnp.linalg.det(walker_dn[:nocc_b,:nocc_b])
    # <exp(T1)HF|walker>/<HF|walker>
    t1 = jnp.linalg.det(wave_data["mo_ta"].T.conj() @ walker_up
        ) * jnp.linalg.det(wave_data["mo_tb"].T.conj() @ walker_dn) / o0
    t2 = gt2g * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>

    return t1, t2, e0, e1